## Cleaning decisions log — people_analytics_CLEAN.csv

1. Stripped whitespace from employee_id before deduplication
2. Removed 24 duplicate rows — kept first occurrence
3. Standardised 8 department name variants to 8 clean values
4. Converted hire_date and exit_date to datetime
5. Nulled 5 future hire dates (2026-03-15 — bulk entry error)
6. Nulled 8 invalid performance scores (outside 1.0–5.0 range)
7. Cleared exit_date for active employees with incorrect dates
8. Flagged 15 leavers missing exit_date (dq_missing_exit_date)
9. Imputed numeric nulls with department median
10. Filled gender and ethnicity nulls with 'Not disclosed'

In [ ]:
# Load libraries and view the dataset
import pandas as pd
import numpy as np
people= pd.read_csv('/content/people_analytics_RAW.csv')
people.head(10)

,employee_id,department,job_level,location,gender,ethnicity,hire_date,tenure_months,performance_score,engagement_score,absence_days_ytd,left_company,exit_reason,exit_date
0,EMP01683,Technology,Manager,London,Male,White,2022-08-30,28,4.0,3.9,14.0,1,Compensation,2022-12-16
1,EMP01746,People & Culture,Senior Analyst,London,Male,White,2023-04-01,21,2.7,1.0,10.0,0,NaN,NaN
2,EMP01735,Sales & Commercial,Manager,London,Male,White,2021-10-29,38,2.6,3.4,0.0,1,Better opportunity,2022-01-20
3,EMP01326,Customer Success,Analyst,London,Male,Black / African / Caribbean,2023-12-31,12,2.2,2.3,6.0,1,Better opportunity,2024-03-23
4,EMP01731,Technology,Senior Analyst,Amsterdam,Male,White,2022-04-30,32,3.9,2.8,11.0,0,NaN,NaN
5,EMP02033,Sales & Commercial,Senior Manager,Dublin,Female,Asian / Asian British,2020-05-29,55,3.7,NaN,9.0,0,NaN,NaN
6,EMP01246,Legal & Compliance,Analyst,London,Female,Black / African / Caribbean,2023-12-01,13,2.3,1.8,16.0,0,NaN,NaN
7,EMP01117,Customer Success,Senior Analyst,London,Male,White,2023-08-31,16,2.4,5.0,10.0,0,NaN,NaN
8,EMP01379,Customer Success,Manager,London,Male,White,2022-05-31,31,3.4,5.0,7.0,0,NaN,NaN
9,EMP01237,Sales & Commercial,Senior Analyst,Lagos,Female,NaN,2023-06-01,19,3.2,4.4,4.0,0,NaN,NaN


In [ ]:
# See the details of the dataset
print(people.shape)

(1224, 14)


In [ ]:
print(people.dtypes)

employee_id           object
department            object
job_level             object
location              object
gender                object
ethnicity             object
hire_date             object
tenure_months          int64
performance_score    float64
engagement_score     float64
absence_days_ytd     float64
left_company           int64
exit_reason           object
exit_date             object
dtype: object


In [ ]:
# View missing values in the dataset
print(people.isnull().sum())

employee_id            0
department             0
job_level              0
location               0
gender                24
ethnicity             73
hire_date              0
tenure_months          0
performance_score     36
engagement_score      48
absence_days_ytd      61
left_company           0
exit_reason          990
exit_date            995
dtype: int64


In [ ]:
# DATA CLEANING- Duplicate Detection
ws_count = (people['employee_id'] != people['employee_id'].str.strip()).sum()
print(f"employee_id rows with whitespace: {ws_count}")

# Check duplicate count BEFORE stripping
dupes_before = people.duplicated(subset='employee_id').sum()
print(f"Duplicates detected BEFORE stripping whitespace: {dupes_before}")

# Strip the whitespace
people['employee_id'] = people['employee_id'].str.strip()

# Check duplicate count AFTER stripping (accurate count)
dupes_after = people.duplicated(subset='employee_id').sum()
print(f"Duplicates detected AFTER stripping whitespace: {dupes_after}")

employee_id rows with whitespace: 20
Duplicates detected BEFORE stripping whitespace: 23
Duplicates detected AFTER stripping whitespace: 24


In [ ]:
# DATA CLEANING- Duplicate Removal
# Numbers of rows before removal
rows_before=len(people)
# Remove duplicates
people = people.drop_duplicates(subset='employee_id', keep='first')
# Row count after duplicate removal
rows_after = len(people)

print(f"Rows before removal: {rows_before}")
print(f"Rows after removal: {rows_after}")
print(f"Duplicates removed: {rows_before - rows_after}")



Rows before removal: 1224
Rows after removal: 1200
Duplicates removed: 24


In [ ]:
# DATA CLEANING- Department Names Standardisation
# See the status of all departments
people['department']. value_counts()

,count
department,
Technology,254
Sales & Commercial,218
Operations,203
Customer Success,158
Finance,114
People & Culture,79
Legal & Compliance,66
Marketing,59
OPERATIONS,13


In [ ]:
dept_corrections={'technology': 'Technology',
                  'TECHNOLOGY': 'Technology',
                  'Tech': 'Technology',
                  'finance': 'Finance',
                  'Finance': 'Finance',
                  'Finanace':'Finance',
                  'Ops': 'Operations',
                  'operations': 'Operations',
                  'OPERATIONS':'Operations'
                  }
people['department'] = people['department'].replace(dept_corrections)

# Verify the new values we have
people['department'].value_counts()

,count
department,
Technology,275
Operations,221
Sales & Commercial,218
Customer Success,158
Finance,124
People & Culture,79
Legal & Compliance,66
Marketing,59


In [ ]:
# Convert Hire Date and Exit Date to Datetime
people['hire_date']=pd.to_datetime(people['hire_date'], format='%Y-%m-%d')
people['exit_date']=pd.to_datetime(people['exit_date'], format='%Y-%m-%d')

# Check for future hire dates (end date is 31st December 2024)
dataset_end=pd.Timestamp ('2024-12-31')
future_date=people[people['hire_date']>dataset_end]

print(f'Future hire dates: {len(future_date)}')
print(future_date[['employee_id','hire_date']])

Future hire dates: 5
     employee_id  hire_date
554     EMP02129 2026-03-15
604     EMP02114 2026-03-15
698     EMP01775 2026-03-15
1020    EMP01146 2026-03-15
1069    EMP02167 2026-03-15


In [ ]:
# Save the index of future date rows BEFORE nulling hire_date
# We do this first because once hire_date is set to NaT we cannot distinguish these
# 5 rows from other legitimate nulls
future_idx = people[people['hire_date'] > dataset_end].index

# Null both hire_date and tenure_months for those 5 rows tenure_months is
# derived from hire_date so it is also unreliable
people.loc[future_idx, 'hire_date'] = pd.NaT
people.loc[future_idx, 'tenure_months'] = np.nan
#Confirm there is no more future dates
remaining=len(people[people['hire_date']>dataset_end])
print(f'Future hire dates remaining: {remaining}')
print(f"Null hire_date :{people['hire_date'].isna().sum()}")

Future hire dates remaining: 0
Null hire_date :5


In [ ]:
# Fix Invalid Performance Scores
# Performance score is between 1.00 and 5.00
# Check the invalid scores that exist
invalid_perf = people[
    (people['performance_score'] < 1.0) |
    (people['performance_score'] > 5.0)
]

print(f"Invalid performance scores found: {len(invalid_perf)}")
print(invalid_perf[['employee_id', 'department', 'job_level', 'performance_score']])

Invalid performance scores found: 8
     employee_id          department       job_level  performance_score
101     EMP01982    Customer Success         Analyst                5.5
271     EMP01631    Customer Success        Director                0.0
369     EMP01309    Customer Success         Manager               -1.0
503     EMP01693          Operations  Senior Manager                5.5
567     EMP01417  Sales & Commercial        Director               -1.0
775     EMP01393  Sales & Commercial         Analyst                0.0
974     EMP01131          Operations  Senior Analyst                0.0
1105    EMP02077  Sales & Commercial         Manager                6.0


In [ ]:
# Convert invalid performance scores to null
people.loc[(people['performance_score']>5.0) | (people['performance_score']<1.0), 'performance_score']=np.nan
invalid_remaining = ((people['performance_score']<1.0) | (people['performance_score']>5.0)).sum()
print(f"Invalid performance scores remaining: {invalid_remaining}")
print(f"Null performance scores: {people['performance_score'].isna().sum()}")

Invalid performance scores remaining: 0
Null performance scores: 43


In [ ]:
# Fix Logical mismatch between left_company and exit_date
 # Leavers without exit date= Case 1
 # Active employees with exit date= Case 2
# We have to flag leavers with no exit date for attrition rate
# and clear the exit date attached to active employees

# Case 1- Leavers without exit date
Case_1=(people['left_company'] == 1) & (people['exit_date'].isna())
print(f"Case 1 — leavers with no exit date: {Case_1.sum()}")

# Case 2 — active employees with an exit date
Case_2 = (people['left_company'] == 0) & (people['exit_date'].notna())
print(f"Case_2 — active employees with exit date: {Case_2.sum()}")

# Fix Case 2 — clear the incorrect exit dates
people.loc[Case_2, 'exit_date'] = pd.NaT

# Add a data quality flag for Case 1
people['dq_missing_exit_date'] = Case_1.astype(int)

# Verify Case 2 is resolved
Case_2_remaining = ((people['left_company'] == 0) & (people['exit_date'].notna())).sum()
print(f"Case_2_remaining after fix: {Case_2_remaining}")
print(f"Rows flagged for missing exit date: {people['dq_missing_exit_date'].sum()}")

# Note: Case 2 showed 0 — no active employees with exit dates.
# Likely removed during the removal of duplicates.
# Case 1: 15 leavers have no exit date — flagged, not dropped.
# Their left_company = 1 is still valid for attrition analysis.

Case 1 — leavers with no exit date: 15
Case_2 — active employees with exit date: 10
Case_2_remaining after fix: 0
Rows flagged for missing exit date: 15


In [ ]:
# Fix Missing Values
# Numerical columns will be filled with the department median
# Categorical columns will be filled with undisclosed

# Numeric columns
for col in ['performance_score', 'engagement_score', 'absence_days_ytd']:
    people[col] = people.groupby('department')[col].transform(
        lambda x: x.fillna(x.median())
    )

# Categorical columns
people['gender']=people['gender'].fillna('Not disclosed')
people['ethnicity'] = people['ethnicity'].fillna('Not disclosed')

# Verify all columns should now show 0 nulls
print(people[['performance_score', 'engagement_score',
          'absence_days_ytd', 'gender', 'ethnicity']].isnull().sum())

performance_score    0
engagement_score     0
absence_days_ytd     0
gender               0
ethnicity            0
dtype: int64


In [ ]:
# FEATURE ENGINEERING
# Tenure Bands
people['tenure_band'] = pd.cut(
    people['tenure_months'],
    bins=[0, 6, 12, 24, 36, 60, 9999],
    labels=['0-6m', '7-12m', '13-24m', '25-36m', '37-60m', '60m+'],
    right=True
)

# Check the distribution across bands
print("Tenure Band Distribution")
print(people['tenure_band'].value_counts().sort_index())

Tenure Band Distribution
tenure_band
0-6m      143
7-12m     131
13-24m    305
25-36m    251
37-60m    263
60m+      102
Name: count, dtype: int64


In [ ]:
# Salary Proxy
# We do not have actual salaries in our dataset, hence we applied the market midpoint based
# on job level. These benchmarks are the UK market rate for this sector (Source: CIPD)
salary_proxy = {
    'Analyst':        38000,
    'Senior Analyst': 52000,
    'Manager':        68000,
    'Senior Manager': 88000,
    'Director':       118000
}

people['salary_proxy'] = people['job_level'].map(salary_proxy)

# Replacement cost estimate
# Industry standard- replacing an employee costs approximately 60% of their annual salary
# (recruitment fees, onboarding, lost productivity during the vacancy period).
# This gives us a £ cost per leaver for Q2.

people['replacement_cost_est'] = people['salary_proxy'] * 0.60

# Check both columns look correct
print("SALARY PROXY AND REPLACEMENT COST BY LEVEL")
print(people.groupby('job_level')[['salary_proxy','replacement_cost_est']]
      .first()
      .sort_values('salary_proxy'))


SALARY PROXY AND REPLACEMENT COST BY LEVEL
                salary_proxy  replacement_cost_est
job_level                                         
Analyst                38000               22800.0
Senior Analyst         52000               31200.0
Manager                68000               40800.0
Senior Manager         88000               52800.0
Director              118000               70800.0


In [ ]:
# Flight Risk Score
# This combines low engagement and low performance into a single risk indicator per employee
# Engagement is weighted 60% because it is the a leading indicator and 40% for performance
# Formular (5-engagement)*60% + (5-performance)* 40%
people['flight_risk_score'] = (
    (5 - people['engagement_score']) * 0.6 +
    (5 - people['performance_score']) * 0.4
).round(2)

# Flag high flight risk employees
# The  75th percentile threshold means roughly 25% of employees will be flagged for intervention
threshold = people['flight_risk_score'].quantile(0.75)
people['high_risk_employee'] = (people['flight_risk_score'] >= threshold).astype(int)

print(f"Flight risk score range: {people['flight_risk_score'].min():.2f} to {people['flight_risk_score'].max():.2f}")
print(f"75th percentile threshold: {threshold:.2f}")
print(f"Employees flagged as high flight risk: {people['high_risk_employee'].sum()}")
print(f"High risk as % of workforce: {people['high_risk_employee'].mean()*100:.1f}%")

Flight risk score range: 0.08 to 3.72
75th percentile threshold: 2.02
Employees flagged as high flight risk: 302
High risk as % of workforce: 25.2%


In [ ]:
# Hire Year
people['hire_year'] = people['hire_date'].dt.year

# Check the distribution — how many hires per year
print("HIRES BY YEAR")
print(people['hire_year'].value_counts().sort_index())
print(f"\nNull hire_years : {people['hire_year'].isna().sum()}")

HIRES BY YEAR
hire_year
2016.0      2
2017.0      5
2018.0     40
2019.0     64
2020.0    109
2021.0    164
2022.0    253
2023.0    313
2024.0    245
Name: count, dtype: int64

Null hire_years : 5


In [ ]:
# Missing exit date
# Flags leavers (left_company=1) who have no exit_date recorded.
print(f"Total leavers:             {people['left_company'].sum()}")
print(f"Leavers missing exit date: {people['dq_missing_exit_date'].sum()}")
print(f"Leavers with exit date:    {people['left_company'].sum() - people['dq_missing_exit_date'].sum()}")

Total leavers:             229
Leavers missing exit date: 15
Leavers with exit date:    214


In [ ]:
# Expot Clean Dataset for Power BI
# Convert tenure band from categorical to string because  pandas categorical dtype does not
# export cleanly to CSV.
people['tenure_band'] = people['tenure_band'].astype(str)

# Final Audit before export
print("FINAL COLUMN CHECK")
print(f"Rows: {len(people)}  |  Columns: {len(people.columns)}")

print("\nNull counts — cleaned columns only:")
cols_to_check = ['department', 'performance_score', 'engagement_score',
                 'absence_days_ytd', 'gender', 'ethnicity',
                 'tenure_band', 'salary_proxy', 'flight_risk_score']
print(people[cols_to_check].isnull().sum())

print("\nNew columns created:")
new_cols = ['tenure_band', 'salary_proxy', 'replacement_cost_est',
            'flight_risk_score', 'high_risk_employee',
            'hire_year', 'dq_missing_exit_date']

# Save the clean dataset as a CSV file
people.to_csv('people_analytics_CLEAN.csv', index=False)

print("Export complete: people_analytics_CLEAN.csv")
print(f"Rows: {len(people)}")
print(f"Columns: {list(people.columns)}")

FINAL COLUMN CHECK
Rows: 1200  |  Columns: 21

Null counts — cleaned columns only:
department           0
performance_score    0
engagement_score     0
absence_days_ytd     0
gender               0
ethnicity            0
tenure_band          0
salary_proxy         0
flight_risk_score    0
dtype: int64

New columns created:
Export complete: people_analytics_CLEAN.csv
Rows: 1200
Columns: ['employee_id', 'department', 'job_level', 'location', 'gender', 'ethnicity', 'hire_date', 'tenure_months', 'performance_score', 'engagement_score', 'absence_days_ytd', 'left_company', 'exit_reason', 'exit_date', 'dq_missing_exit_date', 'tenure_band', 'salary_proxy', 'replacement_cost_est', 'flight_risk_score', 'high_risk_employee', 'hire_year']


In [ ]:
# FLIGHT RISK MODEL VALIDATION
# We check whether employees flagged as high flight risk
# actually left at a higher rate than unflagged employees.
# This validates whether the composite score is predictive
# or just noise.
# ============================================================

# Overall attrition rate — our baseline
overall_rate = people['left_company'].mean()

# Attrition rate among HIGH flight risk employees
high_risk = people[people['high_risk_employee'] == 1]
high_risk_attrition = high_risk['left_company'].mean()

# Attrition rate among LOW flight risk employees
low_risk = people[people['high_risk_employee'] == 0]
low_risk_attrition = low_risk['left_company'].mean()

# How many flagged employees actually left
flagged_and_left = len(people[
    (people['high_risk_employee'] == 1) &
    (people['left_company'] == 1)
])

print("FLIGHT RISK MODEL VALIDATION")
print("=" * 45)
print(f"Total employees:              {len(people):,}")
print(f"Flagged as high risk:         {people['high_risk_employee'].sum():,}")
print(f"Flagged AND left:             {flagged_and_left:,}")
print(f"")
print(f"Overall attrition rate:       {overall_rate*100:.1f}%")
print(f"High risk attrition rate:     {high_risk_attrition*100:.1f}%")
print(f"Low risk attrition rate:      {low_risk_attrition*100:.1f}%")
print(f"")
print(f"Flight risk conversion:       {high_risk_attrition*100:.1f}%")
print(f"Lift over baseline:           {high_risk_attrition/overall_rate:.2f}x")


FLIGHT RISK MODEL VALIDATION
Total employees:              1,200
Flagged as high risk:         302
Flagged AND left:             110

Overall attrition rate:       19.1%
High risk attrition rate:     36.4%
Low risk attrition rate:      13.3%

Flight risk conversion:       36.4%
Lift over baseline:           1.91x
